# SIMBES — Notebook M1: Análisis Nodal e IPR

**Objetivo pedagógico**: entender la intersección IPR (yacimiento) y VLP (bomba) y cómo el VSD desplaza el punto de operación.

**Física cubierta**: Darcy (lineal), Vogel (bifásico), Leyes de Afinidad.

---

In [ ]:
import sys
sys.path.insert(0, '../backend')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, FloatSlider, IntSlider

from physics.ipr import calc_aof, ipr_q_to_pwf, ipr_pwf_to_q, find_operating_point
from physics.hydraulics import pump_head_ft, pump_bep, affinity_laws

plt.style.use('dark_background')
print('✓ Módulos cargados correctamente')

## 1. Parámetros base del pozo

In [ ]:
# ── Parámetros de yacimiento ──────────────────────────────────
Pr    = 3500   # psi  — Presión estática del reservorio
Pb    = 1800   # psi  — Presión de burbuja
IP    = 1.5    # STB/d/psi — Índice de Productividad

# ── Geometría del pozo ────────────────────────────────────────
depth = 7000   # ft   — Profundidad de la bomba
Pwh   = 150    # psi  — Presión de cabezal
grad  = 0.38   # psi/ft — Gradiente del fluido

# ── VSD ───────────────────────────────────────────────────────
freq  = 60.0   # Hz

# ── Cálculos derivados ────────────────────────────────────────
aof = calc_aof(Pr, Pb, IP)
qb  = IP * max(Pr - Pb, 0)

print(f'AOF = {aof:,.0f} STB/d')
print(f'Qb (límite Darcy) = {qb:,.0f} STB/d')
print(f'BEP a {freq} Hz = {pump_bep(freq)["Q_bep"]:,.0f} STB/d')

## 2. Construcción de la Curva IPR

La curva IPR tiene dos zonas:
- **Zona Darcy** (Pwf ≥ Pb): relación lineal Q = IP × (Pr − Pwf)
- **Zona Vogel** (Pwf < Pb): relación cuadrática, el gas libre reduce la movilidad del petróleo

In [ ]:
def build_curves(Pr, Pb, IP, depth, Pwh, freq, grad):
    """Construye curvas IPR y VLP para graficar."""
    aof   = calc_aof(Pr, Pb, IP)
    Q_arr = np.linspace(0, aof * 1.1, 300)
    
    ipr_pwf = np.array([ipr_q_to_pwf(q, Pr, Pb, IP) for q in Q_arr])
    
    def vlp(Q):
        pump_psi    = pump_head_ft(Q, freq) * grad
        static_psi  = grad * depth
        friction_psi = 1.4e-5 * Q**2
        return max(0, Pwh + static_psi - pump_psi + friction_psi)
    
    vlp_pwf = np.array([vlp(q) for q in Q_arr])
    op      = find_operating_point(Pr, Pb, IP, vlp)
    
    return Q_arr, ipr_pwf, vlp_pwf, op

Q_arr, ipr_pwf, vlp_pwf, op = build_curves(Pr, Pb, IP, depth, Pwh, freq, grad)
print(f'Punto de operación: Q = {op["Q"]:,.0f} STB/d  |  Pwf = {op["Pwf"]:,.0f} psi')
print(f'Drawdown: {(Pr - op["Pwf"]) / Pr * 100:.1f}% de Pr')

In [ ]:
def plot_nodal(Pr, Pb, IP, depth, Pwh, freq, grad):
    Q_arr, ipr_pwf, vlp_pwf, op = build_curves(Pr, Pb, IP, depth, Pwh, freq, grad)
    aof  = calc_aof(Pr, Pb, IP)
    bep  = pump_bep(freq)["Q_bep"]
    
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.set_facecolor('#0D1424')
    fig.patch.set_facecolor('#0B0F1A')
    
    # Zonas IPR
    Q_darcy = Q_arr[ipr_pwf >= Pb]
    Q_vogel = Q_arr[ipr_pwf < Pb]
    ax.plot(Q_darcy, ipr_pwf[ipr_pwf >= Pb], color='#38BDF8', lw=2.5, label='IPR (Darcy — lineal)')
    ax.plot(Q_vogel, ipr_pwf[ipr_pwf < Pb],  color='#818CF8', lw=2.5, label='IPR (Vogel — bifásico)', linestyle='--')
    
    # VLP
    ax.plot(Q_arr, vlp_pwf, color='#34D399', lw=2.5, linestyle='-.', label=f'VLP — Bomba BES ({freq} Hz)')
    
    # Pb
    ax.axhline(Pb, color='#FBBF24', lw=1.2, linestyle=':', alpha=0.8)
    ax.text(aof * 0.85, Pb + 50, f'Pb = {Pb:,} psi', color='#FBBF24', fontsize=9)
    
    # BEP
    ax.axvline(bep, color='#F472B6', lw=1.0, linestyle=':', alpha=0.7)
    ax.text(bep + 20, Pr * 0.95, f'BEP\n{bep:,.0f}', color='#F472B6', fontsize=8, ha='left')
    
    # Punto de operación
    if op:
        ax.scatter(op['Q'], op['Pwf'], color='#FB7185', s=120, zorder=10, label='Punto de Operación')
        ax.annotate(
            f"  Q = {op['Q']:,} STB/d\n  Pwf = {op['Pwf']:,} psi",
            xy=(op['Q'], op['Pwf']), xytext=(op['Q'] + aof*0.04, op['Pwf'] + Pr*0.03),
            color='#FB7185', fontsize=9,
            arrowprops=dict(arrowstyle='->', color='#FB7185', lw=1)
        )
    
    ax.set_xlabel('Caudal Q (STB/d)', color='#94A3B8', fontsize=11)
    ax.set_ylabel('Presión Fluyente de Fondo Pwf (psi)', color='#94A3B8', fontsize=11)
    ax.set_title(f'Análisis Nodal — Pr={Pr:,} psi  |  Pb={Pb:,} psi  |  IP={IP} STB/d/psi  |  {freq} Hz',
                 color='#E2E8F0', fontsize=12, pad=12)
    ax.tick_params(colors='#64748B')
    ax.spines['bottom'].set_color('#1E293B')
    ax.spines['left'].set_color('#1E293B')
    ax.spines['top'].set_color('#1E293B')
    ax.spines['right'].set_color('#1E293B')
    ax.grid(color='#1E293B', linewidth=0.7)
    ax.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1', fontsize=10)
    ax.set_xlim(0, aof * 1.05)
    ax.set_ylim(0, Pr * 1.1)
    plt.tight_layout()
    plt.show()

plot_nodal(Pr, Pb, IP, depth, Pwh, freq, grad)

## 3. Exploración Interactiva — Leyes de Afinidad

Observa cómo la variación de frecuencia del VSD desplaza la curva VLP y mueve el punto de operación a lo largo de la curva IPR.

In [ ]:
@interact(
    Pr=IntSlider(value=3500, min=500, max=7000, step=100, description='Pr (psi)'),
    Pb=IntSlider(value=1800, min=100, max=3500, step=100, description='Pb (psi)'),
    IP=FloatSlider(value=1.5, min=0.1, max=10.0, step=0.1, description='IP (STB/d/psi)'),
    freq=FloatSlider(value=60.0, min=30.0, max=70.0, step=1.0, description='Frecuencia (Hz)'),
)
def interactive_nodal(Pr, Pb, IP, freq):
    Pb = min(Pb, Pr - 50)
    plot_nodal(Pr, Pb, IP, depth=7000, Pwh=150, freq=freq, grad=0.38)

## 4. Análisis de Sensibilidad — Impacto de la Frecuencia

Barrido de frecuencias de 40 a 65 Hz para visualizar cómo varía la producción.

In [ ]:
freqs     = np.arange(40, 68, 2)
Q_ops     = []
Pwf_ops   = []

for f in freqs:
    def vlp_f(Q, f=f):
        return max(0, Pwh + grad*depth - pump_head_ft(Q, f)*grad + 1.4e-5*Q**2)
    op = find_operating_point(Pr, Pb, IP, vlp_f)
    Q_ops.append(op['Q'] if op else 0)
    Pwf_ops.append(op['Pwf'] if op else 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0B0F1A')

for ax in axes:
    ax.set_facecolor('#0D1424')
    ax.tick_params(colors='#64748B')
    ax.grid(color='#1E293B', linewidth=0.7)
    for spine in ax.spines.values():
        spine.set_color('#1E293B')

axes[0].plot(freqs, Q_ops, 'o-', color='#38BDF8', lw=2.5, ms=7)
axes[0].axvline(60, color='#F472B6', lw=1, linestyle='--', alpha=0.7)
axes[0].set_xlabel('Frecuencia VSD (Hz)', color='#94A3B8')
axes[0].set_ylabel('Q Operativo (STB/d)', color='#94A3B8')
axes[0].set_title('Producción vs. Frecuencia', color='#E2E8F0')

axes[1].plot(freqs, Pwf_ops, 's-', color='#34D399', lw=2.5, ms=7)
axes[1].axhline(Pb, color='#FBBF24', lw=1, linestyle=':', alpha=0.8)
axes[1].text(freqs[0]+0.5, Pb+30, f'Pb={Pb:,}', color='#FBBF24', fontsize=9)
axes[1].set_xlabel('Frecuencia VSD (Hz)', color='#94A3B8')
axes[1].set_ylabel('Pwf Operativo (psi)', color='#94A3B8')
axes[1].set_title('Pwf vs. Frecuencia', color='#E2E8F0')

plt.tight_layout()
plt.show()

print('\n--- Tabla de resultados ---')
print(f'{"Freq (Hz)":>10}  {"Q (STB/d)":>12}  {"Pwf (psi)":>12}  {"Drawdown %":>12}')
for f, q, p in zip(freqs, Q_ops, Pwf_ops):
    dd = (Pr - p) / Pr * 100 if p > 0 else 100
    print(f'{f:>10.0f}  {q:>12,.0f}  {p:>12,.0f}  {dd:>12.1f}')